In [ ]:
%pip install python-dotenv langchain langchain-classic langchain_core langchain-tavily langchain-community langchain-openai openai langchain-google-genai langchain-anthropic dataclasses

In [ ]:
from dataclasses import dataclass
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

anthropic_model = ChatAnthropic(model="claude-sonnet-4-5", max_retries=3, temperature=0.7)

agent = create_agent(
    anthropic_model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant."
)

result = agent.invoke(
    {"messages": [
        {"role": "system", "content": """You are a financial assistant.
            you should give the account information from the user database.
            response with balance with all user details in json format"""},
        {"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user123")
)

print(result)

print("=========")
print(result["messages"][-1].content)